# Kandinsky V2A ses kalitesi karşılaştırması
## Lite Distill ve tam Lite

Bu notebook **eğitim yapmaz**. Aynı 5 saniyelik video aralığını, aynı ses prompt’unu ve aynı seed’i iki resmi checkpoint ile üretir:

- **A — Lite Distill:** hızlı, 10 çıkarım adımı.
- **B — Lite:** tam model, 50 çıkarım adımı.

Görüntü, prompt, süre ve seed sabit kalır. Dinlerken ses kalitesini ana ölçüt; senkronizasyonu koruma ölçütü olarak değerlendir. Aynı seed karşılaştırmayı daha düzenli yapar, ancak iki checkpoint’in iç üretim adımları farklı olduğu için birebir aynı başlangıç gürültüsünü garanti etmez.

Resmî A100 ölçümünde Lite, Lite Distill’den daha yavaştır (5 saniyelik temel klipte yaklaşık 372 sn ve 79 sn). Bu ölçüm SageAttention ve torch compile kullanıyor; bu notebook SDPA kullanacağından süreler farklı olabilir. Kalite artışı önceden garanti değildir.

**Önerilen çalışma:** A100 80 GB; aynı video ve kısa prompt; 5 saniye. API anahtarı veya eğitim verisi gerekmez.

## 1 — GPU, Python ve disk ön kontrolü

Tam Lite için A100 80 GB kullan. Bu hücre GPU, Python sürümü ve iki checkpoint’i indirmek için boş diski kontrol eder.

In [ ]:
import os, sys, shutil, subprocess, json, time, hashlib, zipfile
from pathlib import Path

print("Python:", sys.version)
try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True, check=False
    )
except FileNotFoundError:
    raise RuntimeError("GPU bulunamadı. Colab > Runtime > Change runtime type > GPU seçip yeniden bağlan.")

print(gpu.stdout or gpu.stderr)
if gpu.returncode != 0:
    raise RuntimeError("GPU denetlenemedi; GPU runtime'ını seçip yeniden bağlan.")
if sys.version_info < (3, 13):
    raise RuntimeError("Resmî Kandinsky kurulum profili Python 3.13 veya 3.14 gerektiriyor.")
if "A100" not in gpu.stdout or "80" not in gpu.stdout:
    raise RuntimeError("Bu karşılaştırma için A100 80 GB runtime seç.")

free_gib = shutil.disk_usage("/content").free / (1024**3)
print(f"/content boş alanı: {free_gib:.1f} GiB")
if free_gib < 40:
    raise RuntimeError("İki Lite checkpoint'i ve çıktılar için en az 40 GiB boş disk bırak.")

## 2 — Kandinsky’nin resmî ortamını kur

Aynı resmî çalışma ortamını kullanıyoruz. Üretim config’inde SageAttention yerine PyTorch SDPA seçeceğiz; böylece Colab’de SageAttention derlemesine bağımlı olmayacağız.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/kandinskylab/kandinsky-6.git
%cd /content/kandinsky-6
!python -m pip -q install uv
!curl -fsSL https://just.systems/install.sh | bash -s -- --to /usr/local/bin
!just setup
print("Kandinsky kaynak commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 3 — İki checkpoint’i indir

Lite Distill mevcut hızlı karşılaştırma kolumuz, Lite ise aynı model ailesinin 50 adımlı tam sürümüdür. İndirme bir kez yapılır; ağırlıklar Colab çalışma alanında kalır.

In [ ]:
!just download lite-distill lite
print("Lite Distill ve tam Lite checkpoint'leri hazır.")

## 4 — Video aralığını ve prompt’u sabitle

Lite Distill denemesinde kullandığın **aynı videoyu**, **aynı kısa prompt’u**, **aynı aralık başlangıcını** ve **aynı seed’i** gir. Notebook iki model için de aynı 5 saniyelik videoyu hazırlar.

Prompt yalnızca duyulması gereken sesi tarif etsin. Görüntüdeki kamera hareketlerini veya atmosferi uzun uzun anlatma; bu deneyin amacı model/checkpoint farkını ölçmek.

In [ ]:
from google.colab import files
from IPython.display import display, Video, Audio

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("Bir MP4 video yüklemedin.")

filename = next(iter(uploaded))
source_video = Path("/content") / Path(filename).name
source_video.write_bytes(uploaded[filename])

# Lite Distill denemende kullandığın değerlerle değiştir.
EVENT_PROMPT = "footsteps"
SEED = 42
WINDOW_START_SECONDS = 0.0
DURATION_SECONDS = 5

probe = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration",
     "-of", "default=noprint_wrappers=1:nokey=1", str(source_video)],
    capture_output=True, text=True
)
if probe.returncode != 0:
    raise RuntimeError("Video okunamadı: " + probe.stderr[-1200:])
source_duration = float(probe.stdout.strip())
window_end = WINDOW_START_SECONDS + DURATION_SECONDS
if WINDOW_START_SECONDS < 0 or window_end > source_duration + 0.02:
    raise ValueError(f"Seçilen pencere videonun dışında. Video {source_duration:.2f}s.")

INPUT_VIDEO = Path("/content/kandinsky_quality_ab_input.mp4")
prep = subprocess.run(
    ["ffmpeg", "-y", "-ss", str(WINDOW_START_SECONDS), "-i", str(source_video),
     "-t", str(DURATION_SECONDS), "-map", "0:v:0", "-an",
     "-c:v", "libx264", "-pix_fmt", "yuv420p", "-r", "24", str(INPUT_VIDEO)],
    capture_output=True, text=True
)
if prep.returncode != 0:
    raise RuntimeError("5 saniyelik giriş aralığı hazırlanamadı: " + prep.stderr[-1500:])

print(f"Video: {filename} ({source_duration:.2f}s)")
print(f"Seçilen aralık: {WINDOW_START_SECONDS:.2f}–{window_end:.2f}s")
print(f"Prompt: {EVENT_PROMPT!r} | Seed: {SEED}")
display(Video(str(INPUT_VIDEO), embed=True))

## 5 — Aynı koşullarla iki üretim yap

Her iki config’te SDPA, prompt genişletme kapalı, aynı 5 saniyelik video, prompt ve seed kullanılır. Checkpoint’in kendi resmî çıkarım reçetesi adım sayısını belirler: Distill 10, tam Lite 50. İki üretim art arda çalışır ve ayrı MP4 dosyalarına kaydedilir.

In [ ]:
KANDINSKY_DIR = Path("/content/kandinsky-6")
OUTPUT_DIR = Path("/content/kandinsky_quality_ab")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def generate_arm(label, checkpoint, offload_strategy):
    config_path = KANDINSKY_DIR / f"quality-{label}.yaml"
    config_path.write_text(
        f"""checkpoint: {checkpoint}
cache:
  mode: none
offload:
  strategy: {offload_strategy}
compile:
  strategy: none
attention:
  engine: sdpa
text_embedder:
  quantized_qwen: false
beautifier:
  name: none
""",
        encoding="utf-8",
    )
    output_mp4 = OUTPUT_DIR / f"{label}.mp4"
    cmd = [
        "just", "generate", EVENT_PROMPT,
        "--config", str(config_path),
        "--attention-engine", "sdpa",
        "--video-file", str(INPUT_VIDEO),
        "--time-length", str(DURATION_SECONDS),
        "--out", str(output_mp4),
        "--seed", str(SEED),
    ]
    print(f"\nÜretim: {label} | checkpoint={checkpoint}")
    started = time.time()
    result = subprocess.run(cmd, cwd=KANDINSKY_DIR, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-5000:])
    if result.stderr:
        print(result.stderr[-5000:])
    if result.returncode != 0:
        raise RuntimeError(f"{label} üretimi başarısız oldu; hata metnini yukarıdan incele.")
    if not output_mp4.is_file():
        raise FileNotFoundError(f"Beklenen MP4 bulunamadı: {output_mp4}")
    elapsed = time.time() - started
    print(f"{label} tamamlandı: {elapsed:.1f} saniye")
    return output_mp4, elapsed

# Lite Distill, A kolu: 10 adım
LITE_DISTILL_MP4, LITE_DISTILL_SECONDS = generate_arm(
    "A_lite_distill", "lite-distill", "module"
)

# Tam Lite, B kolu: 50 adım
LITE_MP4, LITE_SECONDS = generate_arm(
    "B_lite_full", "lite", "none"
)

## 6 — WAV seslerini çıkar ve dinle

WAV dosyaları MP4’lerden aynı biçimde çıkarılır. Böylece karşılaştırmada video görüntüsü veya oynatıcı farkı yerine ses izine odaklanabilirsin. Önce A ve sonra B’yi dinle; mümkünse sesleri körlemesine sırala.

In [ ]:
def extract_wav(input_mp4, output_wav):
    result = subprocess.run(
        ["ffmpeg", "-y", "-i", str(input_mp4), "-vn", "-ac", "2", "-ar", "44100", str(output_wav)],
        capture_output=True, text=True
    )
    if result.returncode != 0 or not output_wav.is_file():
        raise RuntimeError(f"WAV çıkarılamadı: {result.stderr[-1500:]}")
    return output_wav

LITE_DISTILL_WAV = extract_wav(LITE_DISTILL_MP4, OUTPUT_DIR / "A_lite_distill.wav")
LITE_WAV = extract_wav(LITE_MP4, OUTPUT_DIR / "B_lite_full.wav")

print("A — Lite Distill")
display(Audio(filename=str(LITE_DISTILL_WAV)))
print("B — Tam Lite")
display(Audio(filename=str(LITE_WAV)))

## 7 — Dinleme değerlendirmesi ve sonuç paketi

**Ses kalitesi, doğallık, artefakt/gürültü, olay eşleşmesi** puanlarını iki çıktı için ayrı doldur. Senkronizasyon puanını da kaydet; kalite artışı sync’i bozuyor mu kontrol et.

Ölçek: 1 = düşük / belirgin sorun, 5 = yüksek / iyi. Artefakt puanında yön tersidir: **1 = artefakt yok denecek kadar az, 5 = çok belirgin**. Hücre MP4, WAV ve rapor JSON’unu ZIP’e koyup indirir.

In [ ]:
# Dinledikten sonra aşağıdaki puanları doldur.
scores = {
    "A_lite_distill": {
        "audio_quality_1_to_5": None,
        "naturalness_1_to_5": None,
        "artifacts_or_noise_1_to_5": None,
        "event_match_1_to_5": None,
        "sync_1_to_5": None,
        "notes": "",
    },
    "B_lite_full": {
        "audio_quality_1_to_5": None,
        "naturalness_1_to_5": None,
        "artifacts_or_noise_1_to_5": None,
        "event_match_1_to_5": None,
        "sync_1_to_5": None,
        "notes": "",
    },
}

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

report = {
    "experiment": "kandinsky_v2a_quality_lite_vs_lite_distill_v1",
    "input_video": filename,
    "input_video_sha256": sha256(source_video),
    "input_window_seconds": [WINDOW_START_SECONDS, window_end],
    "prompt": EVENT_PROMPT,
    "seed": SEED,
    "duration_seconds": DURATION_SECONDS,
    "prompt_expansion": "disabled",
    "gpu": gpu.stdout.strip(),
    "source_commit": subprocess.check_output(
        ["git", "-C", str(KANDINSKY_DIR), "rev-parse", "HEAD"], text=True
    ).strip(),
    "models": {
        "A_lite_distill": {
            "checkpoint": "kandinskylab/Kandinsky-6.0-Lite-distill-5s-Diffusers",
            "official_inference_steps": 10,
            "generation_seconds": round(LITE_DISTILL_SECONDS, 2),
            "mp4_sha256": sha256(LITE_DISTILL_MP4),
            "wav_sha256": sha256(LITE_DISTILL_WAV),
        },
        "B_lite_full": {
            "checkpoint": "kandinskylab/Kandinsky-6.0-Lite-5s-Diffusers",
            "official_inference_steps": 50,
            "generation_seconds": round(LITE_SECONDS, 2),
            "mp4_sha256": sha256(LITE_MP4),
            "wav_sha256": sha256(LITE_WAV),
        },
    },
    "listening_scores": scores,
}
report_path = OUTPUT_DIR / "quality_ab_report.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

zip_path = Path("/content/kandinsky_v2a_quality_ab.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in [LITE_DISTILL_MP4, LITE_MP4, LITE_DISTILL_WAV, LITE_WAV, report_path]:
        archive.write(path, arcname=path.name)

print("Karşılaştırma paketi hazır:", zip_path)
files.download(str(zip_path))